In [28]:
import json
import pandas as pd
import duckdb
import numpy as np
import glob
from pathlib import Path

# Lendo dados

In [29]:
caminhos_arquivos = glob.glob('dados_brutos/json/*.json')

In [30]:
# 1. LISTAS PARA ACUMULAR OS DADOS
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

print("Iniciando o processamento dos arquivos JSON...")
#caminhos_arquivos = glob.glob('dados_brutos/*.json')

if not caminhos_arquivos:
    print("ERRO: Nenhum arquivo JSON encontrado na pasta 'dados_brutos/'.")
    exit()

# ---------------------------------------------------------
# 2. EXTRAÇÃO E ACHATAMENTO (FLATTEN)
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)
        
        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            continue
            
        # --- PESSOAS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)
        
        # --- BANCAS ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria 
                    if 'membros_banca' in df_temp.columns:
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel   
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA ---
        prod_bib = dados.get('producao_bibliografica', {})
        def add_to_list(chave, lista_destino):
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA ---
        prod_tec = dados.get('producao_tecnica', {})
        def add_to_list_tec(chave, lista_destino):
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES ---
        patentes = dados.get('patentes_registros', {})
        def add_to_list_pat(chave, lista_destino):
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)


# ---------------------------------------------------------
# 3. CONSOLIDAÇÃO EM DATAFRAMES EXPLÍCITOS
# ---------------------------------------------------------
print("Consolidando DataFrames...")

def consolidar(lista):
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames Patentes
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

Iniciando o processamento dos arquivos JSON...
Consolidando DataFrames...


# Tratando dados

## Informações pessoais

In [31]:
df_pessoas.info()

<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id_lattes              39 non-null     str  
 1   nome_completo          39 non-null     str  
 2   nome_citacoes          39 non-null     str  
 3   sexo                   39 non-null     str  
 4   rotulo                 39 non-null     str  
 5   periodo                39 non-null     str  
 6   bolsa_produtividade    39 non-null     str  
 7   endereco_profissional  39 non-null     str  
 8   atualizacao_cv         39 non-null     str  
 9   url                    39 non-null     str  
 10  texto_resumo           39 non-null     str  
dtypes: str(11)
memory usage: 78.0 KB


In [32]:
# 1. Substituir strings vazias e espaços em branco por NaN
# Usa expressão regular para pegar "" ou "   "
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/03/2026,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",05/12/2024,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
2,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",12/03/2026,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
3,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/07/2025,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...
4,0523104569378276,Marcia Helena Costa Fampa,"FAMPA, M. H. C.;FAMPA, M.;FAMPA, MARCIA H.C.;F...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade Federal do Rio de Janeiro, Progra...",07/04/2026,http://lattes.cnpq.br/0523104569378276,Marcia é Professora da Universidade Federal do...
5,2002515486942024,Jayme Luiz Szwarcfiter,"SZWARCFITER, J. L.;Szwarcfiter, Jayme L.;Jayme...",Masculino,* Sem rótulo,NaN,Nível 1A (***,"Universidade Federal do Rio de Janeiro, COPPE ...",05/09/2025,http://lattes.cnpq.br/2002515486942024,Possui graduação em Engenharia Eletrônica pela...
6,9358511568098561,Edmundo Albuquerque de Souza e Silva,"de Souza e Silva, E.;de Souza e Silva, Edmundo...",Masculino,* Sem rótulo,NaN,Nível SR,"Universidade Federal do Rio de Janeiro, Instit...",15/01/2025,http://lattes.cnpq.br/9358511568098561,Edmundo de Souza e Silva é Engenheiro Elétrico...
7,5815607228657970,Henrique Luiz Cukierman,"CUKIERMAN, H. L.;CUKIERMAN, HENRIQUE LUIZ;CUKI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",08/11/2025,http://lattes.cnpq.br/5815607228657970,Possui graduação em Engenharia de Sistemas pel...
8,2704717555047499,Priscila Machado Vieira Lima,"LIMA, P. M. V.;LIMA, PRISCILA M. V.;LIMA, PRIS...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Núcleo...",08/08/2025,http://lattes.cnpq.br/2704717555047499,Possui graduação em Informática pela Universid...
9,5349830056087028,Pedro Henrique González Silva,"González, P.H.;González, Pedro Henrique;GONZAL...",Masculino,* Sem rótulo,NaN,Nível C,"Universidade Federal do Rio de Janeiro, PESC -...",29/04/2026,http://lattes.cnpq.br/5349830056087028,Professor Adjunto na Universidade Federal do R...


In [33]:
# 2. Tratamento da Data de Atualização
# Converte a string '15/10/2025' para um tipo datetime
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'], 
        format='%d/%m/%Y', 
        errors='coerce' # Se tiver uma data bizarra (ex: 99/99/9999), vira nulo em vez de quebrar o script
    )

In [34]:
# 3. Limpeza do campo Rótulo
if 'rotulo' in df_pessoas.columns:
    # Remove o asterisco e espaços em branco nas pontas
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    # Se o rótulo ficou "Sem rótulo", transforma em nulo verdadeiro
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

In [35]:
# 5. Garantia de Tipagem da Chave Primária e Textos Longos
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

In [36]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [37]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [38]:
df_pessoas.head()

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2026-03-30,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2024-12-05,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
2,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,NaN,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",2026-03-12,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
3,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2025-07-30,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...
4,0523104569378276,Marcia Helena Costa Fampa,"FAMPA, M. H. C.;FAMPA, M.;FAMPA, MARCIA H.C.;F...",Masculino,NaN,NaN,Nível 2,"Universidade Federal do Rio de Janeiro, Progra...",2026-04-07,http://lattes.cnpq.br/0523104569378276,Marcia é Professora da Universidade Federal do...


## Tratando orientações

In [137]:
df_orientacoes.head()

,titulo_trabalho,ano_inicio,orientando,tipo_trabalho,instituicao,curso,id_lattes,status,nivel,ano_conclusao
0,Propriedades estruturais de grafos cordais e c...,2025,Rodrigo Fernandes Souto,Tese,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,doutorado,NaN
1,More on set graphs,2022,Bruno Bandeira Monteiro,Tese,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,doutorado,NaN
2,Colorações Seletivas em Grafos,2026,Rafael Paladini Meirelles,Dissertação,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,mestrado,NaN
3,Algoritmos de caminho em grafos,2025,Eduardo Naslausky,Dissertação,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,mestrado,NaN
4,Algoritmo de Caminho Mínimo: uma atividade de ...,2023,Caio de Campos,Trabalho de Conclusão de Curso,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,tcc,NaN


In [138]:
print(df_orientacoes['tipo_trabalho'].value_counts())
print(df_orientacoes['status'].value_counts())
print(df_orientacoes['nivel'].value_counts())
print(df_orientacoes['curso'].value_counts())

tipo_trabalho
Dissertação                       1273
Trabalho de Conclusão de Curso     696
Tese                               643
                                   195
Trabalho                            30
Monografia                          28
Iniciação científica                14
Name: count, dtype: int64
status
concluidas      2649
em_andamento     230
Name: count, dtype: int64
nivel
mestrado                1291
doutorado                648
tcc                      441
iniciacao_cientifica     345
pos_doutorado             81
especializacao            40
outros                    33
Name: count, dtype: int64
curso
    2879
Name: count, dtype: int64


In [134]:
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

In [139]:
import pandas as pd

print("Iniciando o tratamento da tabela de orientações...")

# 1. Tratamento de Strings: Remove espaços duplos e quebras de linha escondidas
colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
for col in colunas_texto:
    df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
    # Se, após limpar os espaços, o campo ficar vazio ou 'nan', preenche com 'Não informado'
    df_orientacoes[col] = df_orientacoes[col].replace({'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'})

# 2. Conversão segura de anos (de float para Int64 com suporte a Nulo)
df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

# 3. Melhoria estética na coluna 'Nível' para os gráficos do Streamlit
mapeamento_nivel = {
    'mestrado': 'Mestrado',
    'doutorado': 'Doutorado',
    'tcc': 'TCC',
    'iniciacao_cientifica': 'Iniciação Científica',
    'pos_doutorado': 'Pós-Doutorado',
    'especializacao': 'Especialização',
    'outros': 'Outros'
}
df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

# 4. Melhoria estética na coluna 'Status' para os gráficos
mapeamento_status = {
    'concluidas': 'Concluída',
    'em_andamento': 'Em Andamento'
}
df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("\nTratamento concluído com sucesso! Verifique a nova estrutura:")
df_orientacoes.info()

print("\nAmostra dos dados tratados:")
display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())

Iniciando o tratamento da tabela de orientações...

Tratamento concluído com sucesso! Verifique a nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 2879 entries, 0 to 2878
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   titulo_trabalho  2879 non-null   str  
 1   ano_inicio       2879 non-null   int64
 2   orientando       2879 non-null   str  
 3   tipo_trabalho    2879 non-null   str  
 4   instituicao      2879 non-null   str  
 5   curso            2879 non-null   str  
 6   id_lattes        2879 non-null   str  
 7   status           2879 non-null   str  
 8   nivel            2879 non-null   str  
 9   ano_conclusao    2649 non-null   Int64
dtypes: Int64(1), int64(1), str(8)
memory usage: 809.0 KB

Amostra dos dados tratados:


,orientando,nivel,status,ano_conclusao
0,Rodrigo Fernandes Souto,Doutorado,Em Andamento,<NA>
1,Bruno Bandeira Monteiro,Doutorado,Em Andamento,<NA>
2,Rafael Paladini Meirelles,Mestrado,Em Andamento,<NA>
3,Eduardo Naslausky,Mestrado,Em Andamento,<NA>
4,Caio de Campos,TCC,Em Andamento,<NA>


## Informações acerca de periódicos publicados

In [39]:
df_bib_artigos.head()

,titulo,ano,autores,revista,volume,numero,paginas,issn,doi,qualis,id_lattes
0,On the (In)Dependence of the Peano Axioms for ...,2021,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",History and Philosophy of Logic,?,,1-19,1464-5149,http://dx.doi.org/10.1080/01445340.2021.1971005,,0211300683784278
1,Short proofs on the structure of general parti...,2021,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",DISCRETE APPLIED MATHEMATICS,303,,8-13,0166-218X,http://dx.doi.org/10.1016/j.dam.2020.09.007,,0211300683784278
2,Transversals of longest paths,2020,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",DISCRETE MATHEMATICS,343,,111717,0012-365X,http://dx.doi.org/10.1016/j.disc.2019.111717,,0211300683784278
3,Intersection of longest paths in graph classes,2020,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",DISCRETE APPLIED MATHEMATICS,281,,96-105,0166-218X,http://dx.doi.org/10.1016/j.dam.2019.03.022,,0211300683784278
4,On Edge-magic Labelings of Forests,2019,"CERIOLI, M. R.; FERNANDES, C. G. ; LEE, O. ; L...",ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,346,,299-307,1571-0661,http://dx.doi.org/10.1016/j.entcs.2019.08.027,,0211300683784278


In [40]:
print("Aplicando tratamentos na tabela 'bib_artigos'...")

if not df_bib_artigos.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da Revista (Periódico)
    if 'revista' in df_bib_artigos.columns:
        # Força maiúsculo e remove espaços extras no início e no fim
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 4. Tratamento do Ano (Garantir que seja número inteiro)
    if 'ano' in df_bib_artigos.columns:
        # errors='coerce' transforma erros (ex: "Sem ano") em NaN
        # Int64 é o tipo inteiro do Pandas que aceita valores nulos
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 5. Tratamento de Título, DOI e ISSN (Apenas remover espaços ocultos)
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 6. Garantir tipagem da chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento da tabela 'bib_artigos' concluído!")
display(df_bib_artigos[['ano', 'revista', 'doi']].head())

Aplicando tratamentos na tabela 'bib_artigos'...
Tratamento da tabela 'bib_artigos' concluído!


,ano,revista,doi
0,2021,HISTORY AND PHILOSOPHY OF LOGIC,http://dx.doi.org/10.1080/01445340.2021.1971005
1,2021,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2020.09.007
2,2020,DISCRETE MATHEMATICS,http://dx.doi.org/10.1016/j.disc.2019.111717
3,2020,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2019.03.022
4,2019,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,http://dx.doi.org/10.1016/j.entcs.2019.08.027


In [41]:
df_bib_artigos.info()

<class 'pandas.DataFrame'>
RangeIndex: 1997 entries, 0 to 1996
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     1997 non-null   str  
 1   ano        1997 non-null   Int64
 2   autores    1997 non-null   str  
 3   revista    1997 non-null   str  
 4   volume     1960 non-null   str  
 5   numero     220 non-null    str  
 6   paginas    1980 non-null   str  
 7   issn       1956 non-null   str  
 8   doi        1516 non-null   str  
 9   qualis     0 non-null      str  
 10  id_lattes  1997 non-null   str  
dtypes: Int64(1), str(10)
memory usage: 666.6 KB


## Tratando informações de eventos

In [47]:
df_bib_trab_congresso.head()

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,Another Calculational Proof of Cantor's Theorem,2022,http://dx.doi.org/10.5753/wbl.2022.223244,"CERIOLI, MÁRCIA R.; FREITAS, RENATA DE ; VIANA...",Workshop Brasileiro de Lógica,,9,,0211300683784278
1,Presenting Basic Graph Logic,2021,http://dx.doi.org/10.1007/978-3-030-86062-2,"CERIOLI, M. R.; SUGUITANI, L. ; VIANA, PETRUCIO",Diagrams,,132-148,,0211300683784278
2,Transversals of Longest Paths,2017,,"CERIOLI, M. R.; FERNANDES, C. G. ; GOMES, R. ;...",Latin and American Algorithms,,,,0211300683784278
3,On the (in)dependence of the Dedekind-Peano ax...,2017,http://dx.doi.org/10.5540/03.2017.005.01.0239,"CERIOLI, MA'RCIA; NOBREGA, HUGO ; SILVEIRA, GU...",CNMAC 2016 XXXVI Congresso Nacional de Matemát...,,,,0211300683784278
4,"L(2, 1)-coloração de k-árvores e grafos com tr...",2015,http://dx.doi.org/10.5540/03.2015.003.01.0241,"BARROS, GABRIEL F. ; POSNER, DANIEL F. D. ; CE...",XXXV CNMAC Congresso Nacional de Matemática Ap...,,,,0211300683784278


In [110]:
df_bib_trab_congresso.info()

<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     3585 non-null   str  
 1   ano        3585 non-null   int64
 2   doi        3585 non-null   str  
 3   autores    3585 non-null   str  
 4   evento     3585 non-null   str  
 5   cidade     3585 non-null   str  
 6   paginas    3585 non-null   str  
 7   isbn       3585 non-null   str  
 8   id_lattes  3585 non-null   str  
dtypes: int64(1), str(8)
memory usage: 1.0 MB


In [111]:
print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da Revista (Periódico)
    if 'evento' in df_bib_trab_congresso.columns:
        # Força maiúsculo e remove espaços extras no início e no fim
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 4. Tratamento do Ano (Garantir que seja número inteiro)
    if 'ano' in df_bib_trab_congresso.columns:
        # errors='coerce' transforma erros (ex: "Sem ano") em NaN
        # Int64 é o tipo inteiro do Pandas que aceita valores nulos
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 5. Tratamento de Título, DOI e ISSN (Apenas remover espaços ocultos)
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 6. Garantir tipagem da chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento da tabela 'bib_artigos' concluído!")
display(df_bib_trab_congresso[['ano', 'evento', 'doi']].head())

Aplicando tratamentos na tabela 'df_bib_trab_congresso'...
Tratamento da tabela 'bib_artigos' concluído!


,ano,evento,doi
0,2022,WORKSHOP BRASILEIRO DE LÓGICA,http://dx.doi.org/10.5753/wbl.2022.223244
1,2021,DIAGRAMS,http://dx.doi.org/10.1007/978-3-030-86062-2
2,2017,LATIN AND AMERICAN ALGORITHMS,NaN
3,2017,CNMAC 2016 XXXVI CONGRESSO NACIONAL DE MATEMÁT...,http://dx.doi.org/10.5540/03.2017.005.01.0239
4,2015,XXXV CNMAC CONGRESSO NACIONAL DE MATEMÁTICA AP...,http://dx.doi.org/10.5540/03.2015.003.01.0241


In [112]:
df_bib_trab_congresso.head(3)

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,Another Calculational Proof of Cantor's Theorem,2022,http://dx.doi.org/10.5753/wbl.2022.223244,"CERIOLI, MÁRCIA R.; FREITAS, RENATA DE ; VIANA...",WORKSHOP BRASILEIRO DE LÓGICA,NaN,9,NaN,0211300683784278
1,Presenting Basic Graph Logic,2021,http://dx.doi.org/10.1007/978-3-030-86062-2,"CERIOLI, M. R.; SUGUITANI, L. ; VIANA, PETRUCIO",DIAGRAMS,NaN,132-148,NaN,0211300683784278
2,Transversals of Longest Paths,2017,NaN,"CERIOLI, M. R.; FERNANDES, C. G. ; GOMES, R. ;...",LATIN AND AMERICAN ALGORITHMS,NaN,NaN,NaN,0211300683784278


## Tratando informações de classificação

### Database de periódicos

In [94]:
import pandas as pd
import numpy as np

print("Épata 1: Carregando e preparando a base completa da Scopus...")
# Carrega o ficheiro original intacto da Scopus
df_scopus_raw = pd.read_excel('periodicos_percentil.xlsx')

# Limpeza de segurança padrão nas chaves de cruzamento
df_scopus_raw['Title'] = df_scopus_raw['Title'].str.upper().str.strip()
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].astype(str).str.replace('-', '', regex=False).str.strip()

# Mapeia TODOS os títulos/ISSNs que possuem alguma subárea de computação
mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].unique())

# Ordena pelo Percentile (do maior para o menor) e remove duplicados de títulos.
# Isto garante que cada revista apareça apenas uma vez, retendo as informações da linha de maior percentil.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

print("Etapa 2: Preparando a base de artigos do Lattes...")
# Limpeza de segurança padrão nas chaves dos artigos
df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()
df_bib_artigos['issn'] = df_bib_artigos['issn'].astype(str).str.replace('-', '', regex=False).str.strip()

print("Etapa 3: Realizando o cruzamento exato (Match)...")
# Seleciona apenas as colunas da Scopus que solicitou para o join
colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

# Match pelo Nome da Revista
df_match_nome = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')

# Match pelo ISSN
df_match_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')

# Consolida os sucessos e remove possíveis duplicados (artigos que deram match por ambos os critérios)
df_sucessos = pd.concat([df_match_nome, df_match_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

# Cria a coluna booleana baseada nas listas completas que mapeamos no início
df_sucessos['Computation Area'] = df_sucessos['Title'].isin(titulos_computacao) | df_sucessos['E-ISSN'].isin(issns_computacao)

print("Etapa 4: Isolando e tratando os artigos sem correspondência (No Match)...")
# Filtra os artigos que ficaram de fora do grupo de sucessos
artigos_com_match = set(df_sucessos['titulo'] + df_sucessos['id_lattes'])
df_falhas = df_bib_artigos[~(df_bib_artigos['titulo'] + df_bib_artigos['id_lattes']).isin(artigos_com_match)].copy()

# Preenche os atributos padrão solicitados para os artigos não encontrados
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0  # <--- Solicitado: 0 para não encontrados
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Computation Area'] = False  # <--- Solicitado: False para não encontrados

print("Etapa 5: Consolidando a tabela final...")
# Empilha os dois blocos de volta
df_artigos_final = pd.concat([df_sucessos, df_falhas], ignore_index=True)

# Garante a tipagem ideal das novas colunas
df_artigos_final['Percentile'] = df_artigos_final['Percentile'].astype(int)
df_artigos_final['Computation Area'] = df_artigos_final['Computation Area'].astype(bool)

print("\n🚀 Tabela final construída com sucesso!")
print(f"Total de linhas: {len(df_artigos_final)}")

# Define a ordem de colunas ideal exibindo primeiro os dados do artigo e depois as métricas solicitadas
colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', # Dados do Artigo
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', # Dados Scopus
    'Computation Area' # Indicador criado
]
df_artigos_final = df_artigos_final[colunas_finais]

# Mostra uma amostra mista contendo sucessos e falhas para validação
display(df_artigos_final.sample(10))

Épata 1: Carregando e preparando a base completa da Scopus...
Etapa 2: Preparando a base de artigos do Lattes...
Etapa 3: Realizando o cruzamento exato (Match)...
Etapa 4: Isolando e tratando os artigos sem correspondência (No Match)...
Etapa 5: Consolidando a tabela final...

🚀 Tabela final construída com sucesso!
Total de linhas: 1992


,id_lattes,titulo,revista,ano,Scopus Source ID,Title,Percentile,Scopus ASJC Code (Sub-subject Area),Scopus Sub-Subject Area,E-ISSN,Computation Area
617,4602221579308599,Local Symmetry in Random Graphs,IEEE TRANSACTIONS ON NETWORK SCIENCE AND ENGIN...,2020,21100372437,IEEE TRANSACTIONS ON NETWORK SCIENCE AND ENGIN...,93,1705,Computer Networks and Communications,23274697,True
1357,2002515486942024,Sources and Sinks in Comparability Graphs,ORDER (DORDRECHT),1997,<NA>,<NA>,0,<NA>,<NA>,<NA>,False
1490,2291334095539768,TUTORIAL CONVIDADO: Pesquisa Operacional em Ép...,PESQUISA OPERACIONAL PARA O DESENVOLVIMENTO,2021,<NA>,<NA>,0,<NA>,<NA>,<NA>,False
37,8154171198308578,Separating the edges of a graph by a linear nu...,ADVANCES IN COMBINATORICS,2023,21101021579,ADVANCES IN COMBINATORICS,92,2607,Discrete Mathematics and Combinatorics,25175599,False
977,8130520066599912,Computational Indicators to Assist Meeting Fac...,GROUP DECISION AND NEGOTIATION,2011,19318,GROUP DECISION AND NEGOTIATION,94,1201,Arts and Humanities (miscellaneous),15729907,False
833,4436183480921146,Análise do Aspecto Combinatório da Localização...,REVISTA CERES,1988,82230,REVISTA CERES,32,3400,Veterinary (all),21773491,False
898,5370222318394867,Improved kernels for Signed Max Cut parameteri...,DISCRETE MATHEMATICS AND THEORETICAL COMPUTER ...,2017,78470,DISCRETE MATHEMATICS AND THEORETICAL COMPUTER ...,37,2607,Discrete Mathematics and Combinatorics,13658050,True
1804,0907883161698484,A strong symmetric formulation for the Min-deg...,ELECTRONIC NOTES IN DISCRETE MATHEMATICS,2016,<NA>,<NA>,0,<NA>,<NA>,<NA>,False
1687,7541486051032916,Web Usability Inspection Technique Based on De...,IET SOFTWARE (PRINT),2009,<NA>,<NA>,0,<NA>,<NA>,<NA>,False
1422,5349830056087028,MAPPING SUPPLY CHAIN STUDIES FROM THE SUSTAINA...,TECNOLOGIA & CULTURA (CEFET/RJ),2021,<NA>,<NA>,0,<NA>,<NA>,<NA>,False


In [98]:
# Cria a coluna indicando se o match ocorreu de forma adequada
# O método .notna() retorna True se existe um ID da Scopus, e False se for nulo (<NA>)
df_artigos_final['match_adequado'] = df_artigos_final['Scopus Source ID'].notna()

# Atualizando a lista de colunas para exibir essa nova no começo
colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', 'match_adequado', # <- Nova coluna aqui
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 
    'Computation Area'
]
df_artigos_final = df_artigos_final[colunas_finais]

print("\nColuna 'match_adequado' adicionada com sucesso!")
display(df_artigos_final[['titulo', 'revista', 'match_adequado', 'Percentile']].sample(10))


Coluna 'match_adequado' adicionada com sucesso!


,titulo,revista,match_adequado,Percentile
960,Towards the Future of Public Health: Roadmappi...,HEALTHCARE,True,73
827,An Approach for the Steiner problem in directe...,ANNALS OF OPERATIONS RESEARCH,True,87
1926,"Modeling, Mining and Analysis of Multi-Relatio...",JOURNAL OF UNIVERSAL COMPUTER SCIENCE (PRINT),False,0
180,On transitive orientations with restricted cov...,INFORMATION PROCESSING LETTERS,True,41
939,A Hybrid Framework for Maritime Surveillance: ...,SENSORS,True,88
1610,Scheduling wireless links by vertex multicolor...,COMPUTER NETWORKS (1999),False,0
206,Even and odd pairs in comparability and in P4-...,DISCRETE APPLIED MATHEMATICS,True,73
91,An overview of MINLP algorithms and their impl...,ANNALS OF OPERATIONS RESEARCH,True,87
1962,Uncertainty quantification in numerical simula...,COMPUTATIONAL GEOSCIENCES (AMSTERDAM),False,0
1074,What is the best grid-map for self-driving car...,EXPERT SYSTEMS WITH APPLICATIONS,True,97


In [101]:
df_artigos_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 1992 entries, 0 to 1991
Data columns (total 12 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   id_lattes                            1992 non-null   str   
 1   titulo                               1992 non-null   str   
 2   revista                              1992 non-null   str   
 3   ano                                  1992 non-null   Int64 
 4   match_adequado                       1992 non-null   bool  
 5   Scopus Source ID                     1266 non-null   object
 6   Title                                1266 non-null   object
 7   Percentile                           1992 non-null   int64 
 8   Scopus ASJC Code (Sub-subject Area)  1266 non-null   object
 9   Scopus Sub-Subject Area              1266 non-null   object
 10  E-ISSN                               721 non-null    object
 11  Computation Area                     1992 non-null   b

In [102]:
print("Renomeando as colunas do DataFrame final...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas = {
    'titulo': 'titulo_artigo',
    'revista': 'titulo_revista_lattes',
    'ano': 'ano_pub',
    'Scopus Source ID': 'id_scopus',
    'Title': 'titulo_revista_scopus',
    'Percentile': 'maior_percentil',
    'Scopus ASJC Code (Sub-subject Area)': 'codigo_area_maior_percentil',
    'Scopus Sub-Subject Area': 'area_maior_percentil',
    'E-ISSN': 'issn',
    'Computation Area': 'computation_area'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_final.rename(columns=mapeamento_colunas, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_final.info()

Renomeando as colunas do DataFrame final...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 1992 entries, 0 to 1991
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_lattes                    1992 non-null   str   
 1   titulo_artigo                1992 non-null   str   
 2   titulo_revista_lattes        1992 non-null   str   
 3   ano_pub                      1992 non-null   Int64 
 4   match_adequado               1992 non-null   bool  
 5   id_scopus                    1266 non-null   object
 6   titulo_revista_scopus        1266 non-null   object
 7   maior_percentil              1992 non-null   int64 
 8   codigo_area_maior_percentil  1266 non-null   object
 9   area_maior_percentil         1266 non-null   object
 10  issn                         721 non-null    object
 11  computation_area             1992 non-null   bool  
dtypes: Int64(

### Database de conferências

In [ ]:
df_google_raw = pd.read_excel('eventos_classificados.xlsx')

Épata 1: Carregando e preparando a base completa da Scopus...


In [108]:
df_google_raw.head(3)

,Sigla,Nome do evento,Estrato
0,AAAI,AAAI Conference on Artificial Intelligence,A1
1,AAMAS,International Conference on Autonomous Agents ...,A1
2,ACCV,Asian Conference on Computer Vision,A1


In [107]:
df_google_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Sigla           781 non-null    str  
 1   Nome do evento  781 non-null    str  
 2   Estrato         781 non-null    str  
dtypes: str(3)
memory usage: 68.5 KB


In [106]:
df_google_raw['Estrato'].value_counts()

Estrato
A3    171
A4    134
A1    110
B4     90
A2     86
B1     78
B2     60
B3     52
Name: count, dtype: int64

In [109]:
print("Substituindo as classificações na coluna 'Estrato'...")

# 1. Cria o dicionário de substituição ('Valor Antigo': 'Valor Novo')
mapeamento_estratos = {
    'B1': 'A5',
    'B2': 'A6',
    'B3': 'A7',
    'B4': 'A8'
}

# 2. Aplica a substituição apenas na coluna 'Estrato'
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)

# 3. Verifica o resultado para garantir que deu certo
print("\nNova distribuição de Estratos:")
display(df_google_raw['Estrato'].value_counts())

Substituindo as classificações na coluna 'Estrato'...

Nova distribuição de Estratos:


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

In [113]:
# Transforma a coluna 'Nome do evento' para letras maiúsculas
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

# Exibe as primeiras linhas para confirmar a alteração
display(df_google_raw.head())

,Sigla,Nome do evento,Estrato
0,AAAI,AAAI CONFERENCE ON ARTIFICIAL INTELLIGENCE,A1
1,AAMAS,INTERNATIONAL CONFERENCE ON AUTONOMOUS AGENTS ...,A1
2,ACCV,ASIAN CONFERENCE ON COMPUTER VISION,A1
3,ACII,INTERNATIONAL CONFERENCE ON AFFECTIVE COMPUTIN...,A2
4,ACISP,AUSTRALASIAN CONFERENCE ON INFORMATION SECURIT...,A4


In [114]:
import pandas as pd
import re

print("1. Preparando os dados para o cruzamento...")
# Criar coluna limpa no dataframe do Lattes para não perder o dado original
df_bib_trab_congresso['evento_limpo'] = df_bib_trab_congresso['evento'].astype(str).str.upper().str.strip()

# Limpeza no dataframe do Google
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper().str.strip()
df_google_raw['Sigla'] = df_google_raw['Sigla'].str.upper().str.strip()

print("2. Tentando Match Exato pelo Nome do Evento...")
# Cruzamento direto: evento == Nome do evento
df_match_exato = pd.merge(
    df_bib_trab_congresso, 
    df_google_raw, 
    left_on='evento_limpo', 
    right_on='Nome do evento', 
    how='inner'
)
df_match_exato['tipo_match'] = 'Exato'

print("3. Isolando os artigos não encontrados para a busca por Sigla...")
# Usamos titulo + id_lattes para identificar unicamente quem já deu match
ids_sucesso_exato = df_match_exato['titulo'] + df_match_exato['id_lattes']
mascara_nao_encontrados = ~(df_bib_trab_congresso['titulo'] + df_bib_trab_congresso['id_lattes']).isin(ids_sucesso_exato)

df_sem_match_exato = df_bib_trab_congresso[mascara_nao_encontrados].copy()

print("4. Executando Busca por Sigla (com proteção de palavra inteira)...")
# Cria um dicionário para busca rápida: { 'AAAI': {'Nome do evento': '...', 'Estrato': 'A1'} }
df_google_siglas_unicas = df_google_raw.dropna(subset=['Sigla']).drop_duplicates(subset=['Sigla'])
dict_siglas = df_google_siglas_unicas.set_index('Sigla')[['Nome do evento', 'Estrato']].to_dict('index')

# Lista de todas as siglas válidas para procurar
lista_siglas = list(dict_siglas.keys())

def buscar_sigla_no_texto(texto):
    if pd.isna(texto) or texto == 'NAN':
        return None
        
    for sigla in lista_siglas:
        if sigla != "":
            # \b significa 'fronteira de palavra'. Garante que 'SAC' só dê match em ' SAC ' e não em 'TRANSACTIONS'
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, texto):
                return sigla
    return None

# Aplica a função de busca no nome do evento dos que sobraram
df_sem_match_exato['sigla_encontrada'] = df_sem_match_exato['evento_limpo'].apply(buscar_sigla_no_texto)

# Separa quem teve sucesso na busca por sigla
df_sucesso_sigla = df_sem_match_exato[df_sem_match_exato['sigla_encontrada'].notna()].copy()

# Traz os dados (Nome e Estrato) usando o mapeamento do dicionário
df_sucesso_sigla['Nome do evento'] = df_sucesso_sigla['sigla_encontrada'].apply(lambda x: dict_siglas[x]['Nome do evento'])
df_sucesso_sigla['Estrato'] = df_sucesso_sigla['sigla_encontrada'].apply(lambda x: dict_siglas[x]['Estrato'])
df_sucesso_sigla['Sigla'] = df_sucesso_sigla['sigla_encontrada']
df_sucesso_sigla['tipo_match'] = 'Por Sigla'

# Limpa colunas auxiliares
df_sucesso_sigla.drop(columns=['sigla_encontrada'], inplace=True)

print("5. Consolidando os que falharam em ambas as tentativas...")
df_falhas_totais = df_sem_match_exato[df_sem_match_exato['sigla_encontrada'].isna()].copy()
df_falhas_totais.drop(columns=['sigla_encontrada'], inplace=True)

# Preenche os vazios de quem não teve match (pode ajustar para 'A8' dependendo da sua regra de negócio)
df_falhas_totais['Nome do evento'] = pd.NA
df_falhas_totais['Estrato'] = 'A8' 
df_falhas_totais['Sigla'] = pd.NA
df_falhas_totais['tipo_match'] = 'Sem Match'

print("6. Empilhando tudo na base final...")
# Junta os 3 blocos: Sucesso Exato, Sucesso Sigla e Falhas
df_artigos_congresso_final = pd.concat([df_match_exato, df_sucesso_sigla, df_falhas_totais], ignore_index=True)

# Remove a coluna temporária de limpeza
df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

# ==========================================
# RESUMO DOS RESULTADOS
# ==========================================
total_originais = len(df_bib_trab_congresso)
qtd_exato = len(df_match_exato)
qtd_sigla = len(df_sucesso_sigla)
qtd_falhas = len(df_falhas_totais)

print(f"\n--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---")
print(f"Total de Artigos (Lattes): {total_originais}")
print(f"✅ Match Exato (Nome): {qtd_exato} ({round((qtd_exato/total_originais)*100, 1)}%)")
print(f"✅ Match por Sigla: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
print(f"❌ Sem Match: {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

# Exibe uma amostra dos que deram match por sigla para validar a qualidade da lógica
display(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Sigla'][['evento', 'Sigla', 'Nome do evento', 'Estrato']].head())

1. Preparando os dados para o cruzamento...
2. Tentando Match Exato pelo Nome do Evento...
3. Isolando os artigos não encontrados para a busca por Sigla...
4. Executando Busca por Sigla (com proteção de palavra inteira)...
5. Consolidando os que falharam em ambas as tentativas...
6. Empilhando tudo na base final...

--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---
Total de Artigos (Lattes): 3585
✅ Match Exato (Nome): 180 (5.0%)
✅ Match por Sigla: 1213 (33.8%)
❌ Sem Match: 2191 (61.1%)


,evento,Sigla,Nome do evento,Estrato
180,LATIN AND AMERICAN ALGORITHMS,LATIN,LATIN AMERICAN THEORETICAL INFORMATICS SYMPOSIUM,A3
181,CNMAC 2016 XXXVI CONGRESSO NACIONAL DE MATEMÁT...,CNMAC,CONGRESSO NACIONAL DE MATEMÁTICA APLICADA E CO...,A8
182,XXXV CNMAC CONGRESSO NACIONAL DE MATEMÁTICA AP...,CNMAC,CONGRESSO NACIONAL DE MATEMÁTICA APLICADA E CO...,A8
183,4TH LATIN-AMERICAN WORKSHOP ON CLIQUES IN GRAPHS,LATIN,LATIN AMERICAN THEORETICAL INFORMATICS SYMPOSIUM,A3
184,4TH LATIN-AMERICAN WORKSHOP ON CLIQUES IN GRAPHS,LATIN,LATIN AMERICAN THEORETICAL INFORMATICS SYMPOSIUM,A3


In [121]:
import pandas as pd
import re

print("1. Preparando dados e ordenando por tamanho do nome...")
df_bib_trab_congresso['evento_limpo'] = df_bib_trab_congresso['evento'].astype(str).str.upper().str.strip()
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].astype(str).str.upper().str.strip()
df_google_raw['Sigla'] = df_google_raw['Sigla'].astype(str).str.upper().str.strip()

# O truque de ouro: ordenar os eventos do nome maior para o menor.
# Evita que um evento de nome curto "roube" o match de um evento mais específico.
df_google_raw['tamanho_nome'] = df_google_raw['Nome do evento'].str.len()
df_google_raw = df_google_raw.sort_values(by='tamanho_nome', ascending=False)

# Transforma a base do Google em uma lista de dicionários para a busca ser ultrarrápida
lista_google = df_google_raw.to_dict('records')

print("2. Aplicando a lógica de match (Nome Contido -> Sigla)...")

def encontrar_melhor_match(evento_lattes):
    if pd.isna(evento_lattes) or evento_lattes == 'NAN':
        return pd.NA, pd.NA, 'A8', 'Sem Match'

    # Tentativa 1: Verifica se o Nome Oficial da tabela Google está CONTIDO no nome digitado no Lattes
    # Como a lista está ordenada por tamanho, ele sempre pega o match mais completo primeiro.
    for google in lista_google:
        nome_oficial = google['Nome do evento']
        if pd.notna(nome_oficial) and nome_oficial != 'NAN' and nome_oficial != "":
            if nome_oficial in evento_lattes:
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Nome (Contido)'

    # Tentativa 2: Se falhar no nome, busca a Sigla protegida por limites de palavra (\b)
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla'

    return pd.NA, pd.NA, 'A8', 'Sem Match'

print("Isso pode levar alguns segundos...")
# Aplica a função de busca
resultados = df_bib_trab_congresso['evento_limpo'].apply(encontrar_melhor_match)

print("3. Consolidando base final...")
df_artigos_congresso_final = df_bib_trab_congresso.copy()

# Extraindo os resultados da função para suas respectivas colunas
df_artigos_congresso_final['Sigla'] = [res[0] for res in resultados]
df_artigos_congresso_final['Nome do evento'] = [res[1] for res in resultados]
df_artigos_congresso_final['Estrato'] = [res[2] for res in resultados]
df_artigos_congresso_final['tipo_match'] = [res[3] for res in resultados]

# Remove colunas auxiliares de limpeza
df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

# ==========================================
# RESUMO DOS RESULTADOS
# ==========================================
total_originais = len(df_artigos_congresso_final)
qtd_nome = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Nome (Contido)'])
qtd_sigla = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Sigla'])
qtd_falhas = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Sem Match'])

print(f"\n--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---")
print(f"Total de Artigos (Lattes): {total_originais}")
print(f"✅ Match por Nome (Contido/Exato): {qtd_nome} ({round((qtd_nome/total_originais)*100, 1)}%)")
print(f"✅ Match por Sigla: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
print(f"❌ Sem Match: {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

# Exibe uma amostra dos matches para você validar
display(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] != 'Sem Match'][['evento', 'Nome do evento', 'Estrato', 'tipo_match']].sample(5))

1. Preparando dados e ordenando por tamanho do nome...
2. Aplicando a lógica de match (Nome Contido -> Sigla)...
Isso pode levar alguns segundos...
3. Consolidando base final...

--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---
Total de Artigos (Lattes): 3585
✅ Match por Nome (Contido/Exato): 833 (23.2%)
✅ Match por Sigla: 869 (24.2%)
❌ Sem Match: 1883 (52.5%)


,evento,Nome do evento,Estrato,tipo_match
258,VIII LATIN-AMERICAN GRAPHS,LATIN AMERICAN THEORETICAL INFORMATICS SYMPOSIUM,A3,Por Sigla
824,2O. WORKSHOP DE DESENVOLVIMENTO RÁPIDO DE APLI...,SIMPÓSIO BRASILEIRO DE QUALIDADE DE SOFTWARE,A3,Por Nome (Contido)
78,SBAC-PAD,INTERNATIONAL SYMPOSIUM ON COMPUTER ARCHITECTU...,A3,Por Sigla
488,WOSES - WORKSHOP UM OLHAR SOCIOTÉCNICO SOBRE A...,SIMPÓSIO BRASILEIRO DE QUALIDADE DE SOFTWARE,A3,Por Nome (Contido)
2387,LATIN-AMERICAN ALGORITHMS GRAPHS AND OPTIMIZAT...,"LATIN-AMERICAN ALGORITHMS, GRAPHS AND OPTIMIZA...",A8,Por Sigla


In [123]:
df_artigos_congresso_final.head()
print(df_artigos_congresso_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   titulo          3585 non-null   str  
 1   ano             3585 non-null   Int64
 2   doi             981 non-null    str  
 3   autores         3585 non-null   str  
 4   evento          3573 non-null   str  
 5   cidade          0 non-null      str  
 6   paginas         2505 non-null   str  
 7   isbn            0 non-null      str  
 8   id_lattes       3585 non-null   str  
 9   Sigla           1702 non-null   str  
 10  Nome do evento  1702 non-null   str  
 11  Estrato         3585 non-null   str  
 12  tipo_match      3585 non-null   str  
dtypes: Int64(1), str(12)
memory usage: 1.3 MB
None


In [124]:
import pandas as pd

print("Padronizando os nomes das colunas de eventos...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas_eventos = {
    'titulo': 'titulo_artigo',
    'evento': 'titulo_evento_lattes',
    'Sigla': 'sigla_evento_google',
    'Nome do evento': 'titulo_evento_google',
    'Estrato': 'estrato'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_congresso_final.rename(columns=mapeamento_colunas_eventos, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_congresso_final.info()

Padronizando os nomes das colunas de eventos...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   titulo_artigo         3585 non-null   str  
 1   ano                   3585 non-null   Int64
 2   doi                   981 non-null    str  
 3   autores               3585 non-null   str  
 4   titulo_evento_lattes  3573 non-null   str  
 5   cidade                0 non-null      str  
 6   paginas               2505 non-null   str  
 7   isbn                  0 non-null      str  
 8   id_lattes             3585 non-null   str  
 9   sigla_evento_google   1702 non-null   str  
 10  titulo_evento_google  1702 non-null   str  
 11  estrato               3585 non-null   str  
 12  tipo_match            3585 non-null   str  
dtypes: Int64(1), str(12)
memory usage: 1.3 MB


In [125]:
print("Removendo as colunas 'cidade' e 'isbn'...")

# O parâmetro errors='ignore' é uma trava de segurança. 
# Se você rodar a célula duas vezes sem querer, ele não vai dar erro reclamando que a coluna já sumiu.
df_artigos_congresso_final.drop(columns=['cidade', 'isbn'], inplace=True, errors='ignore')

print("Colunas removidas com sucesso! Estrutura atualizada:")
df_artigos_congresso_final.info()

Removendo as colunas 'cidade' e 'isbn'...
Colunas removidas com sucesso! Estrutura atualizada:
<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   titulo_artigo         3585 non-null   str  
 1   ano                   3585 non-null   Int64
 2   doi                   981 non-null    str  
 3   autores               3585 non-null   str  
 4   titulo_evento_lattes  3573 non-null   str  
 5   paginas               2505 non-null   str  
 6   id_lattes             3585 non-null   str  
 7   sigla_evento_google   1702 non-null   str  
 8   titulo_evento_google  1702 non-null   str  
 9   estrato               3585 non-null   str  
 10  tipo_match            3585 non-null   str  
dtypes: Int64(1), str(10)
memory usage: 1.2 MB


## Conectando com DuckDB

In [140]:
import duckdb

print("Iniciando persistência no DuckDB com as tabelas de Artigos e Orientações...")
con = duckdb.connect('pesquisadores.duckdb')

# ==========================================
# 1. CRIAÇÃO DOS SCHEMAS E SEQUÊNCIAS
# ==========================================

# Tabela Mãe: Professores
query_cria_pessoas = """
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR
);
"""
con.execute(query_cria_pessoas)

# Sequências para os IDs automáticos das três tabelas filhas
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_periodico;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_conferencia;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_orientacao;")

# Tabela Filha 1: Artigos de Periódicos (Scopus)
query_cria_periodicos = """
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    match_adequado BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_periodicos)

# Tabela Filha 2: Artigos de Conferências/Congressos (Google)
query_cria_conferencias = """
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_conferencias)

# Tabela Filha 3: Orientações (NOVA)
query_cria_orientacoes = """
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_orientacoes)

print("Tabelas criadas com sucesso (ou já existentes).")

# ==========================================
# 2. INSERÇÃO DOS DADOS (Carga via Pandas)
# ==========================================
print("Limpando dados antigos (Filhas primeiro, Mãe depois)...")
# Apagar as 3 filhas antes da mãe para não violar a integridade relacional
con.execute("DELETE FROM tb_artigo_periodico")
con.execute("DELETE FROM tb_artigo_conferencia")
con.execute("DELETE FROM tb_orientacoes")
con.execute("DELETE FROM tb_professores")

print("Inserindo novos dados a partir dos DataFrames Pandas...")

# Inserção da Tabela Mãe (Professores)
if not df_pessoas.empty:
    con.execute("INSERT INTO tb_professores SELECT * FROM df_pessoas")

# Inserção da Tabela Filha 1 (Artigos Periódicos)
if not df_artigos_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, 
            match_adequado, id_scopus, titulo_revista_scopus, maior_percentil, 
            codigo_area_maior_percentil, area_maior_percentil, issn, computation_area
        )
        SELECT 
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, 
            match_adequado, id_scopus, titulo_revista_scopus, maior_percentil, 
            codigo_area_maior_percentil, area_maior_percentil, issn, computation_area 
        FROM df_artigos_final
    """)

# Inserção da Tabela Filha 2 (Artigos Conferência)
if not df_artigos_congresso_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores, 
            titulo_evento_lattes, paginas, sigla_evento_google, 
            titulo_evento_google, estrato, tipo_match
        )
        SELECT 
            id_lattes, titulo_artigo, ano, doi, autores, 
            titulo_evento_lattes, paginas, sigla_evento_google, 
            titulo_evento_google, estrato, tipo_match
        FROM df_artigos_congresso_final
    """)

# Inserção da Tabela Filha 3 (Orientações) - NOVA
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando, 
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT 
            id_lattes, titulo_trabalho, ano_inicio, orientando, 
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

con.close()
print("Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.")

Iniciando persistência no DuckDB com as tabelas de Artigos e Orientações...
Tabelas criadas com sucesso (ou já existentes).
Limpando dados antigos (Filhas primeiro, Mãe depois)...
Inserindo novos dados a partir dos DataFrames Pandas...
Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.
